# 🌙 MoonNeuro — нейро-памятка по законам GTA 5 RP (San-Andreas)

Дообучаем **лёгкую** модель Qwen2.5-0.5B на законах серверов Memphis и Orlando,
затем сжимаем её в формат **GGUF Q4 (~0.4 ГБ)** для запуска на любом ПК через Ollama.

Модель умеет вести диалог, отвечает кратко, понимает уровни розыска (★) и сленг.

**Перед запуском:** `Среда выполнения → Сменить среду выполнения → GPU`.


## 1. Проверка GPU


In [ ]:
import torch, subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), 'Включи GPU: Среда выполнения -> Сменить среду выполнения -> GPU'
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
cap = torch.cuda.get_device_capability(0)[0]
print(f'GPU: {torch.cuda.get_device_name(0)} | {gpu_mem:.0f} GB | cc {cap}.x')


## 2. Клонирование репозитория с GitHub (AloxinBay)


In [ ]:
GITHUB_USER = 'AloxinBay'
REPO_NAME   = 'neuro-laws'
REPO_URL = f'https://github.com/{GITHUB_USER}/{REPO_NAME}.git'
import os, shutil
if os.path.exists(REPO_NAME): shutil.rmtree(REPO_NAME)
!git clone $REPO_URL
%cd $REPO_NAME


## 3. Установка зависимостей
Для модели 0.5B обучаем полностью (без 4-bit) — bitsandbytes не нужен.


In [ ]:
!pip -q install \
  'transformers==4.44.2' \
  'trl==0.9.6' \
  'accelerate==0.33.0' \
  'datasets==2.21.0' \
  'sentencepiece' \
  'regex'
print('Зависимости установлены')


## 4. Парсинг законов и сборка датасета (только русский)


In [ ]:
!python scripts/parse_laws.py --laws-dir laws --out data/laws.json
!python scripts/build_dataset.py --in data/laws.json --out data/train.jsonl


## 5. Конфигурация


In [ ]:
# Лёгкая модель ~0.5B. В GGUF Q4 займёт ~0.4 ГБ.
MODEL_NAME = 'Qwen/Qwen2.5-0.5B-Instruct'

# 0.5B маленькая — можно крупный батч
if gpu_mem >= 22:   BATCH, ACCUM = 32, 1
elif gpu_mem >= 14: BATCH, ACCUM = 16, 1
else:               BATCH, ACCUM = 8, 2
SEQ = 1024
EPOCHS = 3
USE_BF16 = cap >= 8
OUTPUT_DIR = 'outputs/merged'
print(f'model={MODEL_NAME} batch={BATCH} accum={ACCUM} epochs={EPOCHS} bf16={USE_BF16}')


## 6. Загрузка модели (полностью, без квантизации)


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16 if USE_BF16 else torch.float16,
).cuda()
model.config.use_cache = False


## 7. Подготовка датасета (chat-шаблон модели)


In [ ]:
from datasets import load_dataset
raw = load_dataset('json', data_files='data/train.jsonl', split='train')
print('Примеров:', len(raw))

def to_text(ex):
    return {'text': tokenizer.apply_chat_template(
        ex['messages'], tokenize=False, add_generation_prompt=False)}

dataset = raw.map(to_text, remove_columns=raw.column_names)
print(dataset[0]['text'][:500])


## 8. Обучение (полное дообучение)


In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

args = TrainingArguments(
    output_dir='outputs/ckpt',
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=ACCUM,
    gradient_checkpointing=True,
    learning_rate=1e-5,
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,
    logging_steps=25,
    save_strategy='no',
    bf16=USE_BF16, fp16=not USE_BF16,
    report_to='none',
)

trainer = SFTTrainer(
    model=model, args=args, train_dataset=dataset,
    dataset_text_field='text', max_seq_length=SEQ, tokenizer=tokenizer,
    packing=True,
)
trainer.train()

model.config.use_cache = True
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print('Модель сохранена в', OUTPUT_DIR)


## 9. Быстрая проверка


In [ ]:
from transformers import pipeline
chat = pipeline('text-generation', model=model, tokenizer=tokenizer,
                max_new_tokens=200, do_sample=False)
SYSTEM = ('Тебя зовут MoonNeuro. Ты — юридический ассистент по законам GTA 5 RP штата San-Andreas. '
          'Отвечай кратко и по делу. ★ — это уровень розыска. '
          'Понимай сленг: госсник — полицейский, крайм — бандит, тулиться — стрелять.')
def ask(q):
    msgs=[{'role':'system','content':SYSTEM},{'role':'user','content':q}]
    p=tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    print(f'❓ {q}\n{chat(p)[0]["generated_text"][len(p):].strip()}\n'+'-'*50)
for q in ['привет','12.8','наказание за 12.8 ч2','какая статья за убийство','кто такой госсник']:
    ask(q)


## 10. Конвертация в GGUF и сжатие до ~0.4 ГБ (Q4_K_M)
Собираем llama.cpp, конвертируем модель и квантуем.


In [ ]:
import os
if not os.path.exists('llama.cpp'):
    !git clone --depth 1 https://github.com/ggerganov/llama.cpp
!pip -q install -r llama.cpp/requirements.txt
!cmake -S llama.cpp -B llama.cpp/build -DLLAMA_CURL=OFF >/dev/null 2>&1
!cmake --build llama.cpp/build --config Release -j --target llama-quantize >/dev/null 2>&1
print('llama.cpp готов')


In [ ]:
import os
os.makedirs('outputs/gguf', exist_ok=True)
# HF -> GGUF (f16)
!python llama.cpp/convert_hf_to_gguf.py outputs/merged \
    --outfile outputs/gguf/moonneuro-f16.gguf --outtype f16
# f16 -> Q4_K_M (~0.4 ГБ)
!./llama.cpp/build/bin/llama-quantize \
    outputs/gguf/moonneuro-f16.gguf outputs/gguf/moonneuro-q4_k_m.gguf Q4_K_M
import os
mb = os.path.getsize('outputs/gguf/moonneuro-q4_k_m.gguf')/1e6
print(f'Готово: outputs/gguf/moonneuro-q4_k_m.gguf  ({mb:.0f} МБ)')


## 11. Скачать модель
Скачай файл `moonneuro-q4_k_m.gguf` — это вся твоя модель (~0.4 ГБ).


In [ ]:
from google.colab import files
files.download('outputs/gguf/moonneuro-q4_k_m.gguf')
# либо на Google Drive:
# from google.colab import drive; drive.mount('/content/drive')
# !cp outputs/gguf/moonneuro-q4_k_m.gguf /content/drive/MyDrive/


## 12. Как запустить локально (оффлайн)

1. Установи **Ollama**: https://ollama.com/download
2. Положи скачанный `moonneuro-q4_k_m.gguf` рядом с файлом `Modelfile` из репозитория.
3. В терминале:
   ```
   ollama create moonneuro -f Modelfile
   ollama run moonneuro
   ```
Готово — модель отвечает на твоём ПК без интернета.
